# Extract Real SaProt Structure Features

This notebook extracts real 3D structure-aware embeddings from PDB files using SaProt.

**Run this BEFORE training!**

Expected time: 4-6 hours on A40 GPU for 512 structures

## 1. Environment Setup

In [22]:
import sys
import os
from pathlib import Path

# Set project root
PROJECT_ROOT = '/workspace/BioPhysTCR'  # RunPod path
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /workspace/BioPhysTCR


In [23]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA A40
GPU Memory: 47.7 GB


## 2. Check PyTorch Version

SaProt requires PyTorch >= 2.6.0

In [24]:
torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.'))

if torch_version < (2, 6, 0):
    print(f"⚠️  PyTorch {torch.__version__} is too old. Upgrading to 2.6.0...")
    !pip install --upgrade 'torch>=2.6.0' --index-url https://download.pytorch.org/whl/cu124
    print("\n✓ PyTorch upgraded. Please RESTART THE KERNEL and run from cell 1!")
else:
    print(f"✓ PyTorch {torch.__version__} is compatible")

✓ PyTorch 2.6.0+cu124 is compatible


## 3. Install Dependencies

In [25]:
!pip install transformers==4.28.0 sentencepiece biopython huggingface_hub -q


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


## 4. Check PDB Files

In [26]:
data_dir = Path('data')
search_dirs = [
    data_dir / 'raw' / 'TCR_complexes',
    data_dir / 'raw' / 'TCR_complexes_2',
    data_dir / 'raw' / 'tcr_struc',
]

pdb_count = 0
for search_dir in search_dirs:
    if search_dir.exists():
        pdbs = list(search_dir.glob('*.pdb')) + list(search_dir.glob('*.cif'))
        pdb_count += len(pdbs)
        print(f"{search_dir.name}: {len(pdbs)} PDB files")

print(f"\nTotal: {pdb_count} PDB files")

TCR_complexes: 512 PDB files
TCR_complexes_2: 272 PDB files

Total: 784 PDB files


## 5. Run SaProt Extraction

This will take 4-6 hours. You can monitor progress below.

In [27]:
# Import the extraction functions
import pickle
import numpy as np
from tqdm.notebook import tqdm
from typing import Dict, List, Optional
from huggingface_hub import snapshot_download

def find_pdb_files(base_dirs: List[str]) -> Dict[str, str]:
    """Find all PDB files and map PDB ID to file path."""
    pdb_files = {}
    
    for base_dir in base_dirs:
        if not os.path.exists(base_dir):
            continue
        
        for root, dirs, files in os.walk(base_dir):
            for file in files:
                if file.endswith(('.pdb', '.cif')):
                    pdb_id = file.split('.')[0]
                    file_path = os.path.join(root, file)
                    
                    if pdb_id not in pdb_files:
                        pdb_files[pdb_id] = file_path
    
    return pdb_files

def load_saprot_model(device: str = 'cuda'):
    """Load SaProt model - downloads first time, then caches locally."""
    from transformers import EsmTokenizer, EsmForMaskedLM, pipeline
    
    print("Downloading SaProt model (first time only, ~2GB)...")
    
    # Download SaProt to local cache
    model_path = snapshot_download(
        repo_id="westlake-repl/SaProt_650M_AF2",
        cache_dir="/workspace/.cache/saprot"
    )
    
    print(f"Loading SaProt from {model_path}...")
    tokenizer = EsmTokenizer.from_pretrained(model_path)
    model = EsmForMaskedLM.from_pretrained(model_path)
    
    # Remove LM head to get embeddings only
    model.lm_head = torch.nn.Sequential()
    
    model = model.to(device)
    model.eval()
    
    print(f"✓ SaProt model loaded on {device}")
    
    # Create feature extraction pipeline
    saprot_pipeline = pipeline('feature-extraction', model=model, tokenizer=tokenizer, device=device)
    
    return saprot_pipeline

def get_structure_sequence(pdb_file: str) -> Optional[str]:
    """Extract structure sequence from PDB file."""
    try:
        from Bio.PDB import PDBParser
        
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('protein', pdb_file)
        
        residues = []
        for model in structure:
            for chain in model:
                for residue in chain:
                    if residue.id[0] == ' ':
                        resname = residue.resname
                        residues.append(resname)
        
        aa_map = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E',
            'PHE': 'F', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
            'LYS': 'K', 'LEU': 'L', 'MET': 'M', 'ASN': 'N',
            'PRO': 'P', 'GLN': 'Q', 'ARG': 'R', 'SER': 'S',
            'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
        }
        
        sequence = ''.join([aa_map.get(res, 'X') for res in residues])
        return sequence if sequence else None
        
    except Exception as e:
        print(f"Error parsing {pdb_file}: {e}")
        return None

def extract_saprot_embedding(
    saprot_pipeline,
    sequence: str
) -> Optional[np.ndarray]:
    """Extract SaProt embedding using pipeline."""
    try:
        # Use pipeline for extraction
        embeddings = saprot_pipeline(sequence)
        
        # Pipeline returns nested list, convert to numpy
        embeddings = np.array(embeddings[0])  # Shape: [seq_len, hidden_dim]
        
        return embeddings
        
    except Exception as e:
        print(f"Error extracting embedding: {e}")
        import traceback
        traceback.print_exc()
        return None

print("Functions loaded ✓")


Functions loaded ✓


In [28]:
# Find PDB files
search_dirs = [
    'data/raw/TCR_complexes',
    'data/raw/TCR_complexes_2',
    'data/raw/tcr_struc',
]

print("Searching for PDB files...")
pdb_files = find_pdb_files(search_dirs)
print(f"Found {len(pdb_files)} unique PDB structures")

Searching for PDB files...
Found 512 unique PDB structures


In [ ]:
# Load SaProt model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
saprot_pipeline = load_saprot_model(device)

ImportError: cannot import name 'cached_property' from 'transformers.utils' (/usr/local/lib/python3.11/dist-packages/transformers/utils/__init__.py)

In [ ]:
# Extract embeddings
output_dir = Path('data/processed/saprot_real')
output_dir.mkdir(parents=True, exist_ok=True)

results = {}
failed = []

print(f"\nProcessing {len(pdb_files)} structures...\n")

for pdb_id, pdb_file in tqdm(pdb_files.items(), desc="Extracting SaProt"):
    try:
        # Get structure sequence
        sequence = get_structure_sequence(pdb_file)
        if sequence is None:
            failed.append((pdb_id, "Failed to parse structure"))
            continue
        
        # Extract SaProt embedding
        embeddings = extract_saprot_embedding(model, tokenizer, sequence, device)
        if embeddings is None:
            failed.append((pdb_id, "Failed to extract embeddings"))
            continue
        
        # Save embedding
        data = {
            'embeddings': embeddings,
            'sequence': sequence,
            'pdb_id': pdb_id,
            'pdb_path': pdb_file,
            'embedding_dim': embeddings.shape[1],
            'seq_len': len(sequence)
        }
        
        output_file = output_dir / f"{pdb_id}.pkl"
        with open(output_file, 'wb') as f:
            pickle.dump(data, f)
        
        results[pdb_id] = data
        
    except Exception as e:
        failed.append((pdb_id, str(e)))

print(f"\n✓ Processed: {len(results)}/{len(pdb_files)}")
print(f"✗ Failed: {len(failed)}")

if failed:
    print("\nFailed structures:")
    for pdb_id, reason in failed[:10]:
        print(f"  {pdb_id}: {reason}")

In [ ]:
# Save summary
summary = {
    'n_processed': len(results),
    'n_failed': len(failed),
    'pdb_ids': list(results.keys()),
    'failed': failed
}

with open(output_dir / 'saprot_extraction_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print("Summary saved ✓")

## 6. Update Combined Features

In [ ]:
combined_features_path = Path('data/processed/combined_features.pkl')

print(f"Updating {combined_features_path}...")

# Load combined features
with open(combined_features_path, 'rb') as f:
    combined = pickle.load(f)

# Load new SaProt embeddings
saprot_data = {}
for pkl_file in output_dir.glob('*.pkl'):
    if pkl_file.name == 'saprot_extraction_summary.pkl':
        continue
    
    pdb_id = pkl_file.stem
    with open(pkl_file, 'rb') as f:
        saprot_data[pdb_id] = pickle.load(f)

# Update combined features
combined['saprot'] = saprot_data

# Update metadata
if 'metadata' in combined and isinstance(combined['metadata'], dict):
    if saprot_data:
        first_key = list(saprot_data.keys())[0]
        combined['metadata']['saprot_dim'] = saprot_data[first_key]['embedding_dim']
    combined['metadata']['n_structures'] = len(saprot_data)

# Save backup
backup_path = combined_features_path.parent / 'combined_features_esm2_backup.pkl'
print(f"Creating backup at {backup_path}")
os.rename(combined_features_path, backup_path)

# Save updated combined features
with open(combined_features_path, 'wb') as f:
    pickle.dump(combined, f)

print(f"✓ Updated combined features with {len(saprot_data)} SaProt embeddings")
print(f"  Embedding dimension: {combined['metadata'].get('saprot_dim', 'unknown')}")

## 7. Verify Results

In [ ]:
# Check a sample embedding
if saprot_data:
    sample_id = list(saprot_data.keys())[0]
    sample = saprot_data[sample_id]
    
    print(f"Sample: {sample_id}")
    print(f"  Sequence length: {sample['seq_len']}")
    print(f"  Embedding shape: {sample['embeddings'].shape}")
    print(f"  Embedding dimension: {sample['embedding_dim']}")
    print(f"\n✓ Real SaProt structure features extracted!")
else:
    print("⚠️  No SaProt data found")

## 8. Next Steps

1. ✓ Real SaProt embeddings extracted
2. Update model config: `saprot_dim` should match the embedding dimension
3. Run `01_data_preparation.ipynb` to create train/val splits
4. Run `02_train_model.ipynb` to start training!

In [ ]:
print("="*60)
print("SaProt extraction complete!")
print("="*60)
print(f"Output directory: {output_dir}")
print(f"Processed: {len(results)} structures")
print(f"Failed: {len(failed)} structures")
print(f"\nSaProt dimension: {combined['metadata'].get('saprot_dim', 'unknown')}")
print("\nYou can now proceed to training!")